# Import Variables

In [2]:
import os
exec(open("./PGS_calc_param.txt").read())

# Run Script

In [3]:
import time
inital_start = time.time()
import subprocess
from pathlib import Path
import pandas as pd
import gzip
from glob import glob
import psutil

##################################################
# 1. Load in weights file from PGS Catalog
##################################################

##### Load in PGS score file from PGS catalog #####
url = f"https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/{PGS_ID}/ScoringFiles/Harmonized/{PGS_ID}_hmPOS_{BUILD}.txt.gz"
file_destination = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_catalog_weights")
file_destination.mkdir(parents=True, exist_ok=True)

raw_file = file_destination / f"{PGS_ID}_hmPOS_{BUILD}.txt.gz"

# Download directly into destination folder if it is not already there
if not raw_file.exists():
    subprocess.run(
        ["curl", "-fL", url, "-o", str(raw_file)],
        check=True
    )

print("Saved score file to:", raw_file)

# Read into enviornment 
score_df = pd.read_csv(
    raw_file,
    sep="\t",
    comment="#",
    compression="gzip"
)


##################################################
# 2. Liftover score file to GRCh38 as needed
##################################################

start = time.time()

##### Extract genome build of score file from meta-data #####
# Although the file name says GRCh38, this is not always true
# and the meta-data is more reliable. 

genome_build = None

with gzip.open(raw_file, "rt") as f:
    for line in f:
        if line.startswith("#genome_build="):
            genome_build = line.strip().split("=")[1]
            break

print("Genome build:", genome_build)

##### Perform liftover if needed #####

needs_liftover = genome_build in ["GRCh37", "hg19"]

if needs_liftover:
    !pip install pyliftover

    print("Lifting over from ", genome_build, "to GRCh38")

    from pyliftover import LiftOver
    lo = LiftOver("hg19", "hg38")

    def liftover_pos(chrom, pos):
        result = lo.convert_coordinate(f"chr{chrom}", int(pos))
        if result:
            new_chr, new_pos, _, _ = result[0]
            return new_chr.replace("chr", ""), new_pos
        return None, None

    score_df[["chr_name_38", "chr_position_38"]] = score_df.apply(
        lambda row: liftover_pos(row["chr_name"], row["chr_position"]),
        axis=1,
        result_type="expand"
    )

    score_df["chr_position_38"] = pd.to_numeric(
        score_df["chr_position_38"],
        errors="coerce"
    ).astype("Int64")

else:
    print("No liftover needed, already in GRCh38")
    # No liftover needed — just copy original coordinates
    score_df["chr_name_38"] = score_df["chr_name"]
    score_df["chr_position_38"] = pd.to_numeric(
        score_df["chr_position"],
        errors="coerce"
    ).astype("Int64")

print(f"Done ({time.time()-start:.1f}s)")

##################################################
# 3.  Prepare SNP IDs to match .bim file
##################################################

bim_start = time.time()
start = time.time()
print("Starting harmonization...")

###########################################################################################
#Grab BIM IDs
###########################################################################################

# Load all BIM files
print("Finding BIM files...")
bim_files = glob("/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr*.bim")

##### track time #####
print(f"Found {len(bim_files)} BIM files ({time.time()-start:.1f}s)")
######################

# Combine bim files into single file
print("Loading BIM files...")
bim = pd.concat(
    [pd.read_csv(f, sep="\t", header=None) for f in bim_files],
    ignore_index=True
)

# Set of BIM SNP IDs for fast lookup
bim_ids = set(bim.iloc[:,1])

##### track time #####
print(f"Loaded {len(bim):,} variants ({time.time()-start:.1f}s)")
######################

##########################################################################################
##Build forward and reverse IDs from PGS catalog score file
##########################################################################################

# Build forward and reverse IDs
print("Building SNP IDs...")
score_df['SNP_ID'] = (
    "chr" + score_df['chr_name'].astype(str)
    + ":" + score_df['chr_position_38'].astype(str)
    + ":" + score_df['effect_allele']
    + ":" + score_df['other_allele']
)

score_df['SNP_rev'] = (
    "chr" + score_df['chr_name'].astype(str)
    + ":" + score_df['chr_position_38'].astype(str)
    + ":" + score_df['other_allele']
    + ":" + score_df['effect_allele']
)

##### track time #####
print(f"Done ({time.time()-start:.1f}s)")
######################

#########################################################################################
### Find allele matches from PGS score file in BIM file
##########################################################################################

# Determine matches
print("Determining matches...")
forward_match = score_df['SNP_ID'].isin(bim_ids)
reverse_match = score_df['SNP_rev'].isin(bim_ids)
both_match = forward_match & reverse_match
flip_mask = (~forward_match) & reverse_match

#########################
# Print statements for QC tracking
#########################

print(f"Forward matches: {forward_match.sum()}")
print(f"Reverse matches: {reverse_match.sum()}")
print(f"Both orientations (Sanity check - should be 0): {both_match.sum()}") 

print(f"Total PRS variants: {len(score_df):,}")
print(f"Total matched variants: {(forward_match | reverse_match).sum()}")

n_found = (forward_match | reverse_match).sum()
n_total = len(score_df)
print(f"Variants found: {n_found:,}/{n_total:,}")
print(f"Percent found: {100 * n_found / n_total:.2f}%")

%env total_variants={n_total}
##### track time #####
print(f"Matches found ({time.time()-start:.1f}s)")
######################

##########################################################################################
#### Create harmonized score file
##########################################################################################
score_df_final = score_df.copy()

# Flip effect weights where necessary
print("Applying allele flips...")
score_df_final.loc[flip_mask, 'effect_weight'] *= -1

# Swap alleles
score_df_final.loc[flip_mask, 'effect_allele'] = score_df.loc[flip_mask, 'other_allele']
score_df_final.loc[flip_mask, 'other_allele'] = score_df.loc[flip_mask, 'effect_allele']

##### track time #####
print(f"Flips applied ({time.time()-start:.1f}s)")
######################

# Use reversed SNP IDs for flipped variants
score_df_final.loc[flip_mask, 'SNP_ID'] = score_df.loc[flip_mask, 'SNP_rev']

n_total = score_df_final['SNP_ID'].isin(bim_ids).sum() 

print(f"Total orientation matches: {n_total}")

#########################################################################################
##### Save new PRS file
##########################################################################################
# Keep only matched variants
score_df_final = score_df_final[forward_match | reverse_match]

# Select columns for PLINK score file
plink_score_df = score_df_final[['SNP_ID', 'effect_allele', 'effect_weight']]

# Save
plink_score_file = f"/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/{PGS_ID}_plink_score_{BUILD}_prepared.txt"
plink_score_df.to_csv(
    plink_score_file,
    sep="\t",
    index=False,
    header=False
)

print(f"Saved {len(plink_score_df):,} variants to:")
print(plink_score_file)

#########################################################################################
##### Save QC metrics file
##########################################################################################
qc_file = f"/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/{PGS_ID}_qc_metrics_{BUILD}.txt"

qc_metrics = {
    "PGS_ID": PGS_ID,
    "BUILD": BUILD,
    "genome_build_from_metadata": genome_build,
    "needs_liftover": needs_liftover,
    "total_pgs_variants": len(score_df),
    "forward_matches": int(forward_match.sum()),
    "reverse_matches": int(reverse_match.sum()),
    "both_orientation_matches": int(both_match.sum()),
    "flipped_variants": int(flip_mask.sum()),
    "matched_variants": int((forward_match | reverse_match).sum()),
    "unmatched_variants": int((~(forward_match | reverse_match)).sum()),
    "percent_found": round(100 * (forward_match | reverse_match).sum() / len(score_df), 2),
    "variants_saved_to_plink_score_file": len(plink_score_df),
    "plink_score_file": plink_score_file,
    "qc_file": qc_file,
    "total_runtime_minutes": round((time.time() - inital_start) / 60, 2),
}

pd.DataFrame([qc_metrics]).to_csv(
    qc_file,
    sep="\t",
    index=False
)

print("Saved QC metrics to:")
print(qc_file)

##### track time #####
print(f"Done; Total time: {(time.time() - inital_start)/60:.1f} minutes")
######################

Saved score file to: /home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_catalog_weights/PGS002308_hmPOS_GRCh38.txt.gz


/tmp/ipykernel_1901/1920625049.py:32: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  score_df = pd.read_csv(


Genome build: hg19
Lifting over from  hg19 to GRCh38
Done (24.3s)
Starting harmonization...
Finding BIM files...
Found 24 BIM files (0.5s)
Loading BIM files...
Loaded 111,404,689 variants (133.2s)
Building SNP IDs...
Done (134.9s)
Determining matches...
Forward matches: 346731
Reverse matches: 911555
Both orientations (Sanity check - should be 0): 0
Total PRS variants: 1,259,754
Total matched variants: 1258286
Variants found: 1,258,286/1,259,754
Percent found: 99.88%
env: total_variants=1259754
Matches found (436.1s)
Applying allele flips...
Flips applied (436.7s)
Total orientation matches: 1258286
Saved 1,258,286 variants to:
/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS002308_plink_score_GRCh38_prepared.txt
Saved QC metrics to:
/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS002308_qc_metrics_GRCh38.txt
Done; Total time: 10.3 minutes
